# run_real_video_vae_latent_probe

00_runtime_identity_and_user_config

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

from paper_workflow.colab_utils.runtime_check import run_runtime_preflight_check
from scripts.prepare_models.prepare_session_autoencoder_kl import prepare_session_autoencoder_kl
from scripts.check_results.check_real_video_vae_latent_outputs import check_real_video_vae_latent_outputs
from scripts.package_results.package_real_video_vae_latent_outputs import package_real_video_vae_latent_outputs
from scripts.package_results.package_real_video_vae_latent_tar_zst import package_real_video_vae_latent_tar_zst

RUN_MODE = 'formal'
RUN_PROFILE = 'formal'
WORKFLOW_KEY = 'real_video_vae_latent_probe_completion_formal'
STEP_KEY = 'step02_run_real_video_vae_latent_probe'
FAMILY_ID = 'family__real_video_vae_latent_probe__freeze001'
PROCESSED_DATASET_KEY = 'davis2017__real_video_vae_latent__256x256__32f__8fps__freeze001'
PROCESSED_DATASET_ROOT = Path('/content/drive/MyDrive/TSTW/datasets/processed') / PROCESSED_DATASET_KEY
FAMILY_ROOT = Path('/content/drive/MyDrive/TSTW/results/families') / FAMILY_ID
RUN_ROOT = Path('/content/TSTW_runtime/runs') / 'real_video_vae_latent_probe_formal'
LOCAL_MODEL_ROOT = Path('/content/TSTW_runtime/session_models/autoencoder_kl')
REQUIRE_FORMAL_PASS = True

01_mount_google_drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

02_read_processed_dataset_registry

03_prepare_local_runtime_workspace

04_clone_or_update_repository

05_install_runtime_dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest', 'diffusers', 'accelerate', 'transformers', 'safetensors', 'lpips', 'pytorch-msssim'], check=True)

06_prepare_session_model

In [ ]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
session_model_manifest = prepare_session_autoencoder_kl(
    model_id='stabilityai/sd-vae-ft-mse',
    local_model_root=LOCAL_MODEL_ROOT,
    revision='main',
    session_manifest_path=RUN_ROOT / 'artifacts' / 'session_model_manifest.json',
)
assert session_model_manifest['model_policy'] == 'session_only_no_drive_model_storage'

07_write_runtime_config

In [ ]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
runtime_config = {
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'step_key': STEP_KEY,
    'local_dataset_root': str(PROCESSED_DATASET_ROOT),
    'dataset_manifest_path': str(PROCESSED_DATASET_ROOT / 'dataset_manifest.json'),
    'runtime_manifest_overrides': {
        'family_root': str(FAMILY_ROOT),
        'session_model_manifest_path': str(RUN_ROOT / 'artifacts' / 'session_model_manifest.json'),
    },
}
runtime_config_path = RUN_ROOT / 'artifacts' / 'runtime_config.json'
runtime_config_path.write_text(json.dumps(runtime_config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

08_check_gpu_and_runtime

In [ ]:
runtime_check_report = run_runtime_preflight_check()
print(runtime_check_report)

09_verify_repository_contract

10_run_smoke_tests

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', '-o', 'addopts=', 'tests/test_real_video_attack_registry_dual_mode.py'], check=True)

11_run_real_video_vae_latent_formal

In [ ]:
formal_runner_command = [
    sys.executable,
    '-m',
    'experiments.real_video_vae_latent_probe.runner',
    '--run-mode', 'formal',
    '--runtime-profile', 'formal',
    '--run-root', str(RUN_ROOT),
    '--protocol-config', 'configs/protocol/real_video_vae_latent_probe.json',
    '--backend-config', 'configs/backend/real_video_vae_latent.json',
    '--attack-matrix', 'configs/attacks/real_video_attack_matrix.json',
    '--ablation-config', 'configs/ablation/real_video_vae_latent_ablation.json',
    '--runtime-config', str(runtime_config_path),
]
subprocess.run(formal_runner_command, check=True)

12_rebuild_tables_and_reports

13_check_real_video_vae_latent_outputs

In [ ]:
formal_validation_summary = check_real_video_vae_latent_outputs(
    run_root=RUN_ROOT,
    run_mode='formal',
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
)
if not formal_validation_summary['status']:
    raise RuntimeError(formal_validation_summary)

14_package_family_results

In [ ]:
compat_pack_root = RUN_ROOT
package_real_video_vae_latent_outputs(run_root=RUN_ROOT, family_root=FAMILY_ROOT)
tar_pack = package_real_video_vae_latent_tar_zst(
    run_root=RUN_ROOT,
    family_root=FAMILY_ROOT,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
)
drive_archive_path = tar_pack['archive_path']
drive_result_summary = {
    'archive_path': str(drive_archive_path),
    'family_summary_path': str(FAMILY_ROOT / 'family_summary.json'),
    'family_checks_path': str(FAMILY_ROOT / 'family_checks.json'),
}
print(drive_result_summary)

15_print_final_family_summary

In [ ]:
result_registry_path = Path('/content/drive/MyDrive/TSTW/registry/result_registry.jsonl')
family_registry_path = Path('/content/drive/MyDrive/TSTW/registry/family_registry.jsonl')
result_registry_entry = {
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'archive_path': str(drive_archive_path),
}
family_summary = {
    'family_id': FAMILY_ID,
    'formal_validation_summary': formal_validation_summary,
    'drive_result_summary': drive_result_summary,
    'result_registry_path': str(result_registry_path),
    'family_registry_path': str(family_registry_path),
}
print(family_summary)